# PyTorch Autograd Overview

## Key Concepts
- Automatic differentiation engine in PyTorch
- Computes gradients for neural network training
- Tracks operations on tensors

## Basic Usage
- Create tensors with `requires_grad=True`
- Perform operations on tensors
- Call `.backward()` to compute gradients
- Access gradients via `.grad` attribute

## Example
```python
x = torch.tensor([1.], requires_grad=True)
y = x * 2
y.backward()
print(x.grad)  # Outputs: tensor([2.])
```

## Important Notes
- Only floating point tensors support gradients
- `.detach()` stops gradient tracking
- `with torch.no_grad():` temporary disables gradient computation
- Use `retain_graph=True` for multiple backward passes

In [1]:
import torch

In [2]:
x = torch.tensor(3.0, requires_grad=True)
x

tensor(3., requires_grad=True)

In [3]:
y = x ** 2
y

tensor(9., grad_fn=<PowBackward0>)

In [4]:
y.backward() # Compute the gradient of y with respect to x

In [5]:
x.grad # dy/dx
# dy/dx = 2 * x, so dy/dx = 2 * 3 = 6

tensor(6.)

In [6]:
x = torch.tensor(3.0, requires_grad=True)
x

tensor(3., requires_grad=True)

In [7]:
y = x ** 2

In [8]:
y

tensor(9., grad_fn=<PowBackward0>)

In [9]:
z = torch.sin(y)
z

tensor(0.4121, grad_fn=<SinBackward0>)

In [10]:
z.backward() # Compute the gradient of z with respect to x
x.grad # dz/dx

tensor(-5.4668)

### Simple NN

#### Manula Way

In [11]:
# Inputs
x_input = torch.tensor(6.7) # Input feature
y_target = torch.tensor(.0) # Target value

w = torch.tensor(0.5) # Weight
b = torch.tensor(0.0) # Bias

In [12]:
# Binary cross entropy loss for scalar values
def binary_cross_entropy_loss(y_target, y_pred):
    eplison = 1e-7 # To avoid log(0)
    y_pred = torch.clamp(y_pred, eplison, 1 - eplison) # Clamp y_pred to avoid log(0)
    # Binary cross entropy loss
    return - (y_target * torch.log(y_pred) + (1 - y_target) * torch.log(1 - y_pred))

In [13]:
# Forward pass
z = w * x_input + b # Linear transformation
y_pred = torch.sigmoid(z) # Sigmoid activation function
loss = binary_cross_entropy_loss(y_target, y_pred) # Loss function
loss

tensor(3.3845)

In [14]:
# Manual backward pass
#1. dL/dy_pred
dL_dy_pred = - (y_target / y_pred) + ((1 - y_target) / (1 - y_pred))
#2. dy_pred/dz
dy_pred_dz = y_pred * (1 - y_pred)
#3. dz/dw
dz_dw = x_input
#4. dz/db
dz_db = 1
#5. dL/dw
dL_dw = dL_dy_pred * dy_pred_dz * dz_dw
#6. dL/db
dL_db = dL_dy_pred * dy_pred_dz * dz_db

In [15]:
print(f"Manual Gradient of loss w.r.t weight (dw): {dL_dw}")
print(f"Manual Gradient of loss w.r.t bias (db): {dL_db}")

Manual Gradient of loss w.r.t weight (dw): 6.472901821136475
Manual Gradient of loss w.r.t bias (db): 0.9661048054695129


### AutoGrad Method

In [16]:
x = torch.tensor(6.7) # Input feature
y_target = torch.tensor(.0) # Target value

In [17]:
w = torch.tensor(0.5, requires_grad=True) # Weight
b = torch.tensor(0.0, requires_grad=True) # Bias

In [18]:
print(f"Initial weight: {w}")
print(f"Initial bias: {b}")

Initial weight: 0.5
Initial bias: 0.0


In [19]:
# Forward pass
z = w * x + b # Linear transformation
y_pred = torch.sigmoid(z) # Sigmoid activation function
y_pred

tensor(0.9661, grad_fn=<SigmoidBackward0>)

In [20]:
loss = binary_cross_entropy_loss(y_target, y_pred) # Loss function
loss

tensor(3.3845, grad_fn=<NegBackward0>)

In [21]:
loss.backward() # Backward pass to compute gradients
print(f"Gradient of loss w.r.t weight (dw): {w.grad}")
print(f"Gradient of loss w.r.t bias (db): {b.grad}")

Gradient of loss w.r.t weight (dw): 6.472901821136475
Gradient of loss w.r.t bias (db): 0.9661048054695129


### Tensor Autograd

In [23]:
tensor_1 = torch.rand(3, 4, requires_grad=True)

In [24]:
tensor_1

tensor([[0.7246, 0.2288, 0.0977, 0.8921],
        [0.6119, 0.0032, 0.8162, 0.7791],
        [0.0673, 0.7018, 0.2255, 0.4131]], requires_grad=True)

In [25]:
y = (tensor_1 ** 2).mean() # Mean of the squared tensor
y

tensor(0.3124, grad_fn=<MeanBackward0>)

In [26]:
y.backward() # Backward pass to compute gradients
tensor_1.grad # Gradient of y with respect to tensor_1

tensor([[0.1208, 0.0381, 0.0163, 0.1487],
        [0.1020, 0.0005, 0.1360, 0.1298],
        [0.0112, 0.1170, 0.0376, 0.0689]])

In [27]:
# Clean up gradients
tensor_1.grad.zero_() # Zero out the gradients
tensor_1.grad # Check if gradients are zeroed out

tensor([[0., 0., 0., 0.],
        [0., 0., 0., 0.],
        [0., 0., 0., 0.]])

### Disable Grad Track

In [28]:
# disable gradient tracking
with torch.no_grad():
    tensor_1 = torch.rand(3, 4)
    tensor_1

In [ ]:
# option_1 = requires_grad_(False)
# option_2 = detach()
# option_3 = context manager
# option_4 = torch.no_grad()  